# 03 — Results Analysis

Reproduce all paper figures and tables from experimental results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load aggregated results
results = {
    'M0':  {'NDCG@10': 0.1133, 'MRR': 0.2343, 'label': 'BPR-MF', 'tier': 1},
    'M1':  {'NDCG@10': 0.1036, 'MRR': 0.2121, 'label': 'LightGCN', 'tier': 1},
    'M1b': {'NDCG@10': 0.1048, 'MRR': 0.2179, 'label': 'SimGCL', 'tier': 1},
    'M1c': {'NDCG@10': 0.1158, 'MRR': 0.2322, 'label': 'XSimGCL', 'tier': 1},
    'M1d': {'NDCG@10': 0.1114, 'MRR': 0.2276, 'label': 'LightGCL', 'tier': 1},
    'M2':  {'NDCG@10': 0.1132, 'MRR': 0.2350, 'label': '+Genome', 'tier': 2},
    'M3':  {'NDCG@10': 0.1133, 'MRR': 0.2355, 'label': '+BERT Title', 'tier': 2},
    'M4':  {'NDCG@10': 0.1185, 'MRR': 0.2421, 'label': '+LLM Profile', 'tier': 2},
    'M5':  {'NDCG@10': 0.1150, 'MRR': 0.2365, 'label': '+Mood', 'tier': 2},
    'M6':  {'NDCG@10': 0.1110, 'MRR': 0.2257, 'label': '+Themes', 'tier': 2},
    'M7':  {'NDCG@10': 0.1187, 'MRR': 0.2452, 'label': '+Prof+Mood', 'tier': 2},
    'M8':  {'NDCG@10': 0.1160, 'MRR': 0.2407, 'label': '+All LLM', 'tier': 2},
    'M9':  {'NDCG@10': 0.1154, 'MRR': 0.2391, 'label': '+Genome+Mood+Themes', 'tier': 2},
    'R2':  {'NDCG@10': 0.1063, 'MRR': 0.2173, 'label': 'RLMRec-gene', 'tier': 3},
    'R3':  {'NDCG@10': 0.1104, 'MRR': 0.2311, 'label': 'KAR', 'tier': 3},
}

df = pd.DataFrame(results).T
df.index.name = 'config'
print(df.sort_values('NDCG@10', ascending=False).to_string())

## Figure: NDCG@10 Bar Chart by Tier

In [ ]:
tier_colors = {1: '#4C72B0', 2: '#55A868', 3: '#C44E52'}
tier_labels = {1: 'Tier 1: Pure CF', 2: 'Tier 2: Content-Augmented', 3: 'Tier 3: LLM-for-RecSys'}

fig, ax = plt.subplots(figsize=(14, 6))

configs = list(results.keys())
ndcg_vals = [results[c]['NDCG@10'] for c in configs]
colors = [tier_colors[results[c]['tier']] for c in configs]
labels = [results[c]['label'] for c in configs]

bars = ax.bar(range(len(configs)), ndcg_vals, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(configs)))
ax.set_xticklabels([f"{c}\n{l}" for c, l in zip(configs, labels)], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('NDCG@10')
ax.set_title('Benchmark Results: NDCG@10 Across All Configurations')

# Add tier legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=tier_colors[t], label=tier_labels[t]) for t in [1, 2, 3]]
ax.legend(handles=legend_elements, loc='upper right')

# Add M7 highlight
best_idx = configs.index('M7')
bars[best_idx].set_edgecolor('red')
bars[best_idx].set_linewidth(2)

plt.tight_layout()
plt.savefig('../docs/figures/ndcg10_bar_chart.pdf', bbox_inches='tight')
plt.show()

## Key Research Question Results

In [ ]:
comparisons = [
    ('Q1: LLM vs Pure CF', 'M4', 'M1', 'LLM content helps beyond CF'),
    ('Q1: LLM vs Best CF', 'M4', 'M1c', 'LLM beats strongest contrastive CF'),
    ('Q2: LLM vs Genome', 'M4', 'M2', 'LLM synthesis > raw genome tags'),
    ('Q3: LLM vs BERT', 'M4', 'M3', 'LLM reasoning > naive encoding'),
    ('Q4: Mood helps', 'M7', 'M4', 'Structured mood adds value'),
    ('Q5: Simple vs RLMRec', 'M7', 'R2', 'Simple injection >> alignment'),
    ('Q5: Simple vs KAR', 'M7', 'R3', 'Simple injection > MoE adapter'),
]

print(f"{'Question':<25} {'Better':>6} {'Worse':>6} {'Δ NDCG@10':>12} {'Rel %':>8}  Interpretation")
print('=' * 100)
for name, better, worse, interp in comparisons:
    b = results[better]['NDCG@10']
    w = results[worse]['NDCG@10']
    delta = b - w
    rel = (b - w) / w * 100
    print(f"{name:<25} {better:>6} {worse:>6} {delta:>+12.4f} {rel:>+7.1f}%  {interp}")